# Phase 7 · Data analysis — Prior Case Retrieval

Computes what the Phase 7 design depends on and writes two small files to send back:
- `P7_analysis/p7_stats.json` contains:
  - the record formats;
  - the number of queries and the size of each candidate pool;
  - **relevant candidates per query** and the best Micro-F1@7 any system could reach with exactly 7 answers;
  - document lengths in sentences, words and BPE tokens, and 512-token chunks per case;
  - masked placeholders in the text;
  - **leakage checks:** a query inside its own pool, overlaps between splits, duplicate texts;
  - how often a candidate is relevant to several queries;
  - a **BM25 reference** (word matching, no neural model) on dev and train, with Micro-P/R/F1@7 and Recall@k.
- `P7_analysis/p7_samples.json` holds a few dev queries, the beginning and end of each, with one relevant candidate and the BM25 hits.

BM25 is used only as a reference score and to plan hard-negative mining. It is never part of a submission.

**Kaggle setup:**
- Accelerator: **CPU** is enough.
- Add-ons → Secrets: **`HF`**. Internet on.
- Add Input → **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer).
- Expected time ≈ 15–30 minutes.
- Download both files from `P7_analysis/` and send them.

In [1]:
# Cell 1 — Imports, folders and settings (everything adjustable is here).
import json, glob, random, hashlib, re, time
from pathlib import Path
from collections import Counter
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

DATA_DIR = Path("/kaggle/working/data")
OUT_DIR = Path("/kaggle/working/P7_analysis")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(42)
K = 7                               # answers per query (fixed by the task)
TOKEN_SAMPLE = 2000                 # candidates per split tokenised with the BPE tokenizer (all queries are tokenised)
TRAIN_BM25_QUERIES = 400            # train queries scored with BM25 (enough for Recall@k estimates)
CHUNK = 512                         # chunk length used by the planned models
print("Outputs:", OUT_DIR)

Outputs: /kaggle/working/P7_analysis


In [2]:
# Cell 2 — Download Phase_7_PCR from Hugging Face using the Kaggle secret "HF".
snapshot_download(repo_id="Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=UserSecretsClient().get_secret("HF"),
                  local_dir=str(DATA_DIR), allow_patterns=["Phase_7_PCR/**"])
P7_DIR = DATA_DIR / "Phase_7_PCR"
for f in sorted(P7_DIR.rglob("*")):
    if f.is_file():
        print(f"{f.relative_to(P7_DIR)}: {f.stat().st_size / 1e6:.1f} MB")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Dev/dev.json: 5.9 MB
Dev/dev_candidates.json: 51.9 MB
Test/test.json: 9.3 MB
Test/test_candidates.json: 71.1 MB
Train/train.json: 33.2 MB
Train/train_candidates.json: 159.8 MB


In [3]:
# Cell 3 — Show the raw structure of every file (type, first keys, one example) before any assumption is made.
RAW = {}
for split, files in {"train": ("Train/train.json", "Train/train_candidates.json"), "dev": ("Dev/dev.json", "Dev/dev_candidates.json"),
                     "test": ("Test/test.json", "Test/test_candidates.json")}.items():
    for kind, rel in zip(("queries", "candidates"), files):
        RAW[(split, kind)] = json.load(open(P7_DIR / rel))

def describe(obj):
    """Short description of a JSON value: type, size and (for records) field names and value types."""
    if isinstance(obj, list):
        first = obj[0] if obj else None
        return {"type": "list", "n": len(obj), "first": describe_record(first)}
    if isinstance(obj, dict):
        k0 = next(iter(obj))
        return {"type": "dict", "n": len(obj), "first_key": k0, "first": describe_record(obj[k0])}
    return {"type": type(obj).__name__}

def describe_record(r):
    if isinstance(r, dict):
        return {k: (f"list[{len(v)}] of {type(v[0]).__name__}" if isinstance(v, list) and v else type(v).__name__) for k, v in r.items()}
    if isinstance(r, list):
        return f"list[{len(r)}] of {type(r[0]).__name__ if r else '?'}"
    return type(r).__name__

stats = {"formats": {f"{s}_{k}": describe(v) for (s, k), v in RAW.items()}}
for name, d in stats["formats"].items():
    print(f"{name:18s} {d}")

train_queries      {'type': 'dict', 'n': 827, 'first_key': '0000000599', 'first': {'text': 'list[54] of str', 'relevant_candidates': 'list[2] of str'}}
train_candidates   {'type': 'dict', 'n': 4320, 'first_key': '0000000262', 'first': {'text': 'list[55] of str'}}
dev_queries        {'type': 'dict', 'n': 118, 'first_key': '0000012637', 'first': {'text': 'list[71] of str', 'relevant_candidates': 'list[11] of str'}}
dev_candidates     {'type': 'dict', 'n': 1023, 'first_key': '0000003060', 'first': {'text': 'list[9] of str'}}
test_queries       {'type': 'dict', 'n': 237, 'first_key': '0000021652', 'first': {'text': 'list[45] of str'}}
test_candidates    {'type': 'dict', 'n': 1727, 'first_key': '0000000987', 'first': {'text': 'list[59] of str'}}


In [4]:
# Cell 4 — Normalise every file to {id: record} with the text joined from its sentences, and the relevant ids for queries.
def as_records(obj):
    """Accepts a list of records with an id field, or a dict id -> record / id -> list of sentences."""
    if isinstance(obj, dict):
        return {str(k): (v if isinstance(v, dict) else {"text": v}) for k, v in obj.items()}
    out = {}
    for r in obj:
        rid = next(r[k] for k in ("id", "query_id", "candidate_id", "case_id", "doc_id") if k in r)
        out[str(rid)] = r
    return out

def sentences(r):
    t = r.get("text", r.get("sentences", ""))
    return [str(s) for s in t] if isinstance(t, list) else [s for s in re.split(r"(?<=[.?!])\s+", str(t)) if s]

REL_KEYS = ("relevant_candidates", "relevant", "labels", "relevant_ids")
QUERIES, POOLS = {}, {}
for split in ("train", "dev", "test"):
    q = as_records(RAW[(split, "queries")])
    QUERIES[split] = {qid: {"sents": sentences(r),
                            "rel": [str(x) for x in next((r[k] for k in REL_KEYS if k in r), [])]} for qid, r in q.items()}
    POOLS[split] = {cid: sentences(r) for cid, r in as_records(RAW[(split, "candidates")]).items()}
    print(f"{split}: {len(QUERIES[split])} queries | pool {len(POOLS[split])} candidates | "
          f"queries with labels {sum(bool(v['rel']) for v in QUERIES[split].values())}")
text = lambda sents: " ".join(sents)
del RAW

train: 827 queries | pool 4320 candidates | queries with labels 817
dev: 118 queries | pool 1023 candidates | queries with labels 118
test: 237 queries | pool 1727 candidates | queries with labels 0


In [5]:
# Cell 5 — Relevant candidates per query, labels missing from the pool, and the best Micro-F1@7 possible with exactly 7 answers.
def pct(xs, ps=(0, 10, 25, 50, 75, 90, 95, 99, 100)):
    arr = np.asarray(xs, dtype=float)
    return {f"p{p}": round(float(np.percentile(arr, p)), 1) for p in ps} | {"mean": round(float(arr.mean()), 2)} if len(arr) else {}

stats["n_queries"] = {s: len(QUERIES[s]) for s in QUERIES}
stats["pool_size"] = {s: len(POOLS[s]) for s in POOLS}
stats["relevant"] = {}
for split in ("train", "dev"):
    rels = [v["rel"] for v in QUERIES[split].values()]
    n_rel = [len(r) for r in rels]
    in_pool = [sum(c in POOLS[split] for c in r) for r in rels]
    hits = sum(min(K, n) for n in in_pool)
    p_max, r_max = hits / (K * len(rels)), hits / max(sum(n_rel), 1)
    stats["relevant"][split] = {
        "per_query": pct(n_rel),
        "distribution": dict(sorted(Counter(min(n, 20) for n in n_rel).items())),   # 20 = "20 or more"
        "share_queries_with_at_most_7": round(float(np.mean([n <= K for n in n_rel])), 3),
        "queries_with_no_relevant": int(sum(n == 0 for n in n_rel)),
        "labels_not_in_pool": int(sum(n_rel) - sum(in_pool)),
        "oracle_at_7": {"precision": round(p_max, 4), "recall": round(r_max, 4), "f1": round(2 * p_max * r_max / max(p_max + r_max, 1e-9), 4)},
        "share_of_pool_relevant_to_some_query": round(len({c for r in rels for c in r} & set(POOLS[split])) / len(POOLS[split]), 3),
    }
    times_relevant = Counter(c for r in rels for c in r)
    stats["relevant"][split]["candidate_relevant_to_n_queries"] = dict(sorted(Counter(min(v, 10) for v in times_relevant.values()).items()))
    stats["relevant"][split]["top_candidates_share_of_labels"] = round(sum(v for _, v in times_relevant.most_common(max(1, len(times_relevant) // 100)))
                                                                       / max(sum(n_rel), 1), 3)   # top 1% most-cited candidates
    print(split, json.dumps(stats["relevant"][split], indent=1))

train {
 "per_query": {
  "p0": 0.0,
  "p10": 1.0,
  "p25": 2.0,
  "p50": 4.0,
  "p75": 9.0,
  "p90": 14.0,
  "p95": 19.0,
  "p99": 31.0,
  "p100": 52.0,
  "mean": 6.42
 },
 "distribution": {
  "0": 10,
  "1": 121,
  "2": 134,
  "3": 100,
  "4": 78,
  "5": 51,
  "6": 44,
  "7": 41,
  "8": 32,
  "9": 27,
  "10": 25,
  "11": 29,
  "12": 21,
  "13": 16,
  "14": 22,
  "15": 11,
  "16": 6,
  "17": 10,
  "18": 6,
  "19": 6,
  "20": 37
 },
 "share_queries_with_at_most_7": 0.7,
 "queries_with_no_relevant": 10,
 "labels_not_in_pool": 0,
 "oracle_at_7": {
  "precision": 0.612,
  "recall": 0.6675,
  "f1": 0.6386
 },
 "share_of_pool_relevant_to_some_query": 0.951,
 "candidate_relevant_to_n_queries": {
  "1": 3363,
  "2": 484,
  "3": 164,
  "4": 44,
  "5": 32,
  "6": 10,
  "7": 2,
  "8": 4,
  "9": 1,
  "10": 3
 },
 "top_candidates_share_of_labels": 0.048
}
dev {
 "per_query": {
  "p0": 1.0,
  "p10": 1.0,
  "p25": 2.0,
  "p50": 5.0,
  "p75": 9.8,
  "p90": 17.3,
  "p95": 24.0,
  "p99": 58.2,
  "p100"

In [6]:
# Cell 6 — Leakage and overlap checks: a query inside its own pool, ids and identical texts shared between splits, duplicates.
key = lambda sents: hashlib.md5(" ".join(" ".join(sents).split()).encode()).hexdigest()
ids = {f"{s}_{k}": set(QUERIES[s] if k == "queries" else POOLS[s]) for s in ("train", "dev", "test") for k in ("queries", "candidates")}
hashes = {f"{s}_queries": {key(v["sents"]) for v in QUERIES[s].values()} for s in QUERIES}
hashes |= {f"{s}_candidates": {key(v) for v in POOLS[s].values()} for s in POOLS}
names = list(ids)
stats["overlap_ids"] = {f"{a}&{b}": len(ids[a] & ids[b]) for i, a in enumerate(names) for b in names[i + 1:] if ids[a] & ids[b]}
stats["overlap_texts"] = {f"{a}&{b}": len(hashes[a] & hashes[b]) for i, a in enumerate(names) for b in names[i + 1:] if hashes[a] & hashes[b]}
stats["query_in_own_pool"] = {s: sum(q in POOLS[s] for q in QUERIES[s]) for s in QUERIES}
stats["query_listed_as_own_relevant"] = {s: sum(q in v["rel"] for q, v in QUERIES[s].items()) for s in ("train", "dev")}
stats["duplicate_texts_within"] = {n: len(ids[n]) - len(hashes[n]) for n in names}
stats["empty_documents"] = {s: {"queries": sum(not v["sents"] for v in QUERIES[s].values()), "candidates": sum(not v for v in POOLS[s].values())}
                            for s in QUERIES}
# Test queries/candidates that also appear, with labels, in train or dev (a Phase-4-style leak).
test_q_texts = {key(v["sents"]) for v in QUERIES["test"].values()}
stats["test_queries_labelled_elsewhere"] = {s: sum(key(v["sents"]) in test_q_texts and bool(v["rel"]) for v in QUERIES[s].values()) for s in ("train", "dev")}
for k in ("overlap_ids", "overlap_texts", "query_in_own_pool", "query_listed_as_own_relevant", "duplicate_texts_within",
          "empty_documents", "test_queries_labelled_elsewhere"):
    print(f"{k}: {stats[k]}")

overlap_ids: {'train_queries&train_candidates': 827, 'train_queries&dev_candidates': 124, 'train_queries&test_candidates': 269, 'train_candidates&dev_queries': 90, 'train_candidates&dev_candidates': 436, 'train_candidates&test_queries': 170, 'train_candidates&test_candidates': 740, 'dev_queries&dev_candidates': 118, 'dev_queries&test_candidates': 43, 'dev_candidates&test_queries': 40, 'dev_candidates&test_candidates': 220, 'test_queries&test_candidates': 237}
overlap_texts: {'train_queries&train_candidates': 827, 'train_queries&dev_candidates': 124, 'train_queries&test_queries': 1, 'train_queries&test_candidates': 270, 'train_candidates&dev_queries': 90, 'train_candidates&dev_candidates': 437, 'train_candidates&test_queries': 171, 'train_candidates&test_candidates': 742, 'dev_queries&dev_candidates': 118, 'dev_queries&test_candidates': 43, 'dev_candidates&test_queries': 40, 'dev_candidates&test_candidates': 222, 'test_queries&test_candidates': 237}
query_in_own_pool: {'train': 827, 'de

In [7]:
# Cell 7 — Lengths in sentences and words (all documents), BPE tokens and 512-token chunks (all queries, a sample of candidates).
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
def token_lengths(texts):
    out = []
    for s in range(0, len(texts), 128):
        out += [len(e.ids) for e in tokenizer.encode_batch(texts[s:s + 128])]
    return np.array(out)

stats["sentences"], stats["words"], stats["tokens"], stats["chunks_512"], stats["fully_within_tokens"] = {}, {}, {}, {}, {}
for split in ("train", "dev", "test"):
    groups = {"queries": [v["sents"] for v in QUERIES[split].values()], "candidates": list(POOLS[split].values())}
    for kind, docs in groups.items():
        name = f"{split}_{kind}"
        stats["sentences"][name] = pct([len(d) for d in docs])
        stats["words"][name] = pct([len(text(d).split()) for d in docs])
        sample = docs if kind == "queries" else random.Random(0).sample(docs, min(TOKEN_SAMPLE, len(docs)))
        tl = token_lengths([text(d) for d in sample])
        stats["tokens"][name] = pct(tl)
        stats["chunks_512"][name] = pct(np.ceil(np.maximum(tl, 1) / CHUNK))
        stats["fully_within_tokens"][name] = {str(k): round(float((tl <= k).mean()), 3) for k in (512, 1024, 2048, 4096, 8192)}
        print(f"{name:17s} words {stats['words'][name]['p50']:>7} (median) | tokens {stats['tokens'][name]} ")

train_queries     words  4508.0 (median) | tokens {'p0': 176.0, 'p10': 1984.0, 'p25': 3264.0, 'p50': 5656.0, 'p75': 9634.5, 'p90': 15632.6, 'p95': 25442.0, 'p99': 55421.1, 'p100': 146729.0, 'mean': 8509.09} 
train_candidates  words  3632.5 (median) | tokens {'p0': 61.0, 'p10': 1129.5, 'p25': 2272.8, 'p50': 4392.5, 'p75': 8237.5, 'p90': 14984.0, 'p95': 23049.9, 'p99': 71751.6, 'p100': 506965.0, 'mean': 7991.69} 
dev_queries       words  4619.5 (median) | tokens {'p0': 723.0, 'p10': 2076.4, 'p25': 3355.5, 'p50': 5615.5, 'p75': 9967.0, 'p90': 19482.9, 'p95': 24562.0, 'p99': 125140.8, 'p100': 130020.0, 'mean': 10469.8} 
dev_candidates    words  4036.0 (median) | tokens {'p0': 45.0, 'p10': 1216.2, 'p25': 2669.0, 'p50': 5053.0, 'p75': 9388.0, 'p90': 20762.6, 'p95': 33615.9, 'p99': 105710.5, 'p100': 506965.0, 'mean': 10720.89} 
test_queries      words  4653.0 (median) | tokens {'p0': 140.0, 'p10': 1491.2, 'p25': 3029.0, 'p50': 5802.0, 'p75': 9861.0, 'p90': 17442.6, 'p95': 24847.4, 'p99': 3679

In [8]:
# Cell 8 — Masked placeholders and citation markers in the text (they decide what the model can match on).
placeholder = re.compile(r"<[A-Za-z_ ]{2,30}>|\[[A-Z_ ]{3,30}\]")
def placeholder_counts(docs, n=1500):
    c, docs_with = Counter(), Counter()
    for d in random.Random(1).sample(docs, min(n, len(docs))):
        found = placeholder.findall(text(d))
        c.update(found); docs_with.update(set(found))
    m = min(n, len(docs))
    return {p: {"per_doc": round(c[p] / m, 2), "share_docs": round(docs_with[p] / m, 3)} for p, _ in c.most_common(12)}

stats["placeholders"] = {"train_queries": placeholder_counts([v["sents"] for v in QUERIES["train"].values()]),
                         "train_candidates": placeholder_counts(list(POOLS["train"].values())),
                         "test_queries": placeholder_counts([v["sents"] for v in QUERIES["test"].values()])}
stats["share_lowercase"] = {n: round(float(np.mean([text(d) == text(d).lower() for d in docs[:1000]])), 3)
                            for n, docs in (("train_queries", [v["sents"] for v in QUERIES["train"].values()]),
                                            ("test_candidates", list(POOLS["test"].values())))}
for k, v in stats["placeholders"].items():
    print(k, v)
print("share of documents entirely lower-case:", stats["share_lowercase"])

train_queries {'<ORG>': {'per_doc': 89.92, 'share_docs': 0.996}, '<NAME>': {'per_doc': 52.5, 'share_docs': 0.989}, '<CITATION>': {'per_doc': 9.67, 'share_docs': 0.999}, '[HARJIT SINGH BEDI]': {'per_doc': 0.0, 'share_docs': 0.001}, '[RAILWAY BOARD]': {'per_doc': 0.0, 'share_docs': 0.001}, '[NTT]': {'per_doc': 0.0, 'share_docs': 0.001}, '[MUPP]': {'per_doc': 0.0, 'share_docs': 0.001}, '[UMPF]': {'per_doc': 0.0, 'share_docs': 0.001}, '[BJP]': {'per_doc': 0.0, 'share_docs': 0.001}, '[VHP]': {'per_doc': 0.0, 'share_docs': 0.001}, '< cm x >': {'per_doc': 0.0, 'share_docs': 0.001}}
train_candidates {'<ORG>': {'per_doc': 92.72, 'share_docs': 0.991}, '<NAME>': {'per_doc': 49.23, 'share_docs': 0.987}, '<CITATION>': {'per_doc': 7.92, 'share_docs': 0.817}, '[SWATANTER KUMAR]': {'per_doc': 0.0, 'share_docs': 0.001}, '[NTT]': {'per_doc': 0.0, 'share_docs': 0.001}, '[MUPP]': {'per_doc': 0.0, 'share_docs': 0.001}, '[UMPF]': {'per_doc': 0.0, 'share_docs': 0.001}, '[BJP]': {'per_doc': 0.0, 'share_docs':

In [9]:
# Cell 9 — BM25 helpers (pure word matching, no neural model) and the Micro-P/R/F1@7 + Recall@k scorer used in every Phase 7 notebook.
def bm25_matrix(pool_texts, k1=1.2, b=0.75, max_features=200_000):
    """Fits the vocabulary on the pool; returns (vectoriser, BM25-weighted pool matrix [docs x terms])."""
    vec = CountVectorizer(lowercase=True, token_pattern=r"(?u)\b[a-z][a-z0-9]+\b", stop_words="english",
                          min_df=2, max_df=0.5, max_features=max_features, dtype=np.float32)
    tf = vec.fit_transform(pool_texts).tocsr()
    n_docs = tf.shape[0]
    df = np.bincount(tf.indices, minlength=tf.shape[1])
    idf = np.log((n_docs - df + 0.5) / (df + 0.5) + 1.0).astype(np.float32)
    dl = np.asarray(tf.sum(1)).ravel()
    norm = k1 * (1 - b + b * dl / dl.mean())
    tf = tf.tocoo()
    w = tf.data * (k1 + 1) / (tf.data + norm[tf.row]) * idf[tf.col]
    return vec, sp.csr_matrix((w, (tf.row, tf.col)), shape=tf.shape)

def rank(scores, pool_ids, query_ids, depth=200):
    """Top `depth` candidate ids per query, never the query itself."""
    pos = {c: i for i, c in enumerate(pool_ids)}
    out = []
    for qi, q in enumerate(query_ids):
        s = scores[qi].copy()
        if q in pos:
            s[pos[q]] = -np.inf
        top = np.argpartition(-s, min(depth, len(s) - 1))[:depth]
        out.append([pool_ids[j] for j in top[np.argsort(-s[top])]])
    return out

def evaluate(ranked, gold, ks=(7, 20, 50, 100, 200)):
    """Micro precision/recall/F1 over the first 7 answers (the official metric) and Recall@k for deeper lists."""
    hits = sum(len(set(r[:K]) & set(g)) for r, g in zip(ranked, gold))
    n_gold = sum(len(g) for g in gold)
    p, r = hits / (K * len(gold)), hits / max(n_gold, 1)
    res = {"micro_p@7": round(p, 4), "micro_r@7": round(r, 4), "micro_f1@7": round(2 * p * r / max(p + r, 1e-9), 4)}
    res |= {f"recall@{k}": round(sum(len(set(x[:k]) & set(g)) for x, g in zip(ranked, gold)) / max(n_gold, 1), 4) for k in ks}
    return res

def bm25_run(split, query_ids, transform=lambda s: text(s)):
    pool_ids = list(POOLS[split])
    vec, W = bm25_matrix([transform(POOLS[split][c]) for c in pool_ids])
    Q = vec.transform([transform(QUERIES[split][q]["sents"]) for q in query_ids])
    Q.data[:] = 1.0                                           # query term presence (long queries)
    scores = (Q @ W.T).toarray()
    return rank(scores, pool_ids, query_ids)
print("BM25 helpers ready")

BM25 helpers ready


In [10]:
# Cell 10 — BM25 reference on dev (all queries) and on a train sample: full text, and which part of the case matters (first / last 2,000 words).
stats["bm25"] = {}
dev_q = [q for q in QUERIES["dev"] if QUERIES["dev"][q]["rel"]]
train_q = random.Random(2).sample([q for q in QUERIES["train"] if QUERIES["train"][q]["rel"]],
                                  min(TRAIN_BM25_QUERIES, len(QUERIES["train"])))
VARIANTS = {"full_text": lambda s: text(s),
            "first_2000_words": lambda s: " ".join(text(s).split()[:2000]),
            "last_2000_words": lambda s: " ".join(text(s).split()[-2000:])}
BM25_DEV = {}
for name, fn in VARIANTS.items():
    t0 = time.time()
    BM25_DEV[name] = bm25_run("dev", dev_q, fn)
    stats["bm25"][f"dev_{name}"] = evaluate(BM25_DEV[name], [QUERIES["dev"][q]["rel"] for q in dev_q])
    print(f"dev   {name:17s} {stats['bm25'][f'dev_{name}']} | {time.time() - t0:.0f} s")
t0 = time.time()
train_ranked = bm25_run("train", train_q)
stats["bm25"]["train_sample_full_text"] = evaluate(train_ranked, [QUERIES["train"][q]["rel"] for q in train_q]) | {"n_queries": len(train_q)}
print(f"train full_text (sample of {len(train_q)}) {stats['bm25']['train_sample_full_text']} | {time.time() - t0:.0f} s")

dev   full_text         {'micro_p@7': 0.1392, 'micro_r@7': 0.1182, 'micro_f1@7': 0.1278, 'recall@7': 0.1182, 'recall@20': 0.1809, 'recall@50': 0.2569, 'recall@100': 0.3638, 'recall@200': 0.5447} | 4 s
dev   first_2000_words  {'micro_p@7': 0.3547, 'micro_r@7': 0.3011, 'micro_f1@7': 0.3257, 'recall@7': 0.3011, 'recall@20': 0.445, 'recall@50': 0.5694, 'recall@100': 0.6578, 'recall@200': 0.7646} | 1 s
dev   last_2000_words   {'micro_p@7': 0.3462, 'micro_r@7': 0.2939, 'micro_f1@7': 0.318, 'recall@7': 0.2939, 'recall@20': 0.4317, 'recall@50': 0.5262, 'recall@100': 0.6064, 'recall@200': 0.7102} | 2 s
train full_text (sample of 400) {'micro_p@7': 0.1139, 'micro_r@7': 0.1191, 'micro_f1@7': 0.1164, 'recall@7': 0.1191, 'recall@20': 0.1859, 'recall@50': 0.2527, 'recall@100': 0.3262, 'recall@200': 0.4278, 'n_queries': 400} | 14 s


In [11]:
# Cell 11 — Where BM25 succeeds and fails on dev: F1@7 by number of relevant cases, rank of the first relevant hit, and TF-IDF cosine for comparison.
ranked = BM25_DEV["full_text"]
by_n = {}
for q, r in zip(dev_q, ranked):
    g = set(QUERIES["dev"][q]["rel"]); n = len(g)
    bucket = "1" if n == 1 else "2-3" if n <= 3 else "4-7" if n <= K else "8-15" if n <= 15 else "16+"
    by_n.setdefault(bucket, []).append((len(set(r[:K]) & g), n))
stats["bm25_dev_by_n_relevant"] = {b: {"queries": len(v), "micro_p@7": round(sum(h for h, _ in v) / (K * len(v)), 4),
                                       "micro_r@7": round(sum(h for h, _ in v) / sum(n for _, n in v), 4)} for b, v in sorted(by_n.items())}
first_hit = [next((i + 1 for i, c in enumerate(r) if c in set(QUERIES["dev"][q]["rel"])), None) for q, r in zip(dev_q, ranked)]
found = [x for x in first_hit if x]
stats["bm25_dev_first_relevant_rank"] = pct(found) | {"queries_without_hit_in_top200": int(sum(x is None for x in first_hit))}
pool_ids = list(POOLS["dev"])
tfidf = TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b[a-z][a-z0-9]+\b", stop_words="english", min_df=2, max_df=0.5,
                        sublinear_tf=True, max_features=200_000, dtype=np.float32)
D = tfidf.fit_transform([text(POOLS["dev"][c]) for c in pool_ids])
Q = tfidf.transform([text(QUERIES["dev"][q]["sents"]) for q in dev_q])
stats["tfidf_cosine_dev"] = evaluate(rank((Q @ D.T).toarray(), pool_ids, dev_q), [QUERIES["dev"][q]["rel"] for q in dev_q])
print("BM25 dev by number of relevant cases:", json.dumps(stats["bm25_dev_by_n_relevant"]))
print("rank of first relevant candidate:", stats["bm25_dev_first_relevant_rank"])
print("TF-IDF cosine dev:", stats["tfidf_cosine_dev"])

BM25 dev by number of relevant cases: {"1": {"queries": 15, "micro_p@7": 0.019, "micro_r@7": 0.1333}, "16+": {"queries": 14, "micro_p@7": 0.1837, "micro_r@7": 0.0426}, "2-3": {"queries": 25, "micro_p@7": 0.1029, "micro_r@7": 0.3158}, "4-7": {"queries": 39, "micro_p@7": 0.1648, "micro_r@7": 0.2133}, "8-15": {"queries": 25, "micro_p@7": 0.1829, "micro_r@7": 0.1199}}
rank of first relevant candidate: {'p0': 1.0, 'p10': 1.0, 'p25': 1.0, 'p50': 4.0, 'p75': 22.0, 'p90': 74.8, 'p95': 118.8, 'p99': 146.9, 'p100': 182.0, 'mean': 21.17, 'queries_without_hit_in_top200': 5}
TF-IDF cosine dev: {'micro_p@7': 0.4007, 'micro_r@7': 0.3402, 'micro_f1@7': 0.368, 'recall@7': 0.3402, 'recall@20': 0.4728, 'recall@50': 0.5827, 'recall@100': 0.668, 'recall@200': 0.7831}


In [12]:
# Cell 12 — Samples to send: three dev queries (start and end), one relevant candidate each, and BM25's top 7 marked hit/miss.
def clip(sents, head=5, tail=3, width=300):
    cut = lambda s: s[:width] + ("…" if len(s) > width else "")
    return {"first": [cut(s) for s in sents[:head]], "last": [cut(s) for s in sents[-tail:]], "n_sentences": len(sents)}

samples = []
for q in random.Random(3).sample(dev_q, min(3, len(dev_q))):
    g = QUERIES["dev"][q]["rel"]
    r = BM25_DEV["full_text"][dev_q.index(q)][:K]
    rel_id = next((c for c in g if c in POOLS["dev"]), None)
    samples.append({"query_id": q, "query": clip(QUERIES["dev"][q]["sents"]), "n_relevant": len(g),
                    "relevant_example": {"id": rel_id, **clip(POOLS["dev"][rel_id])} if rel_id else None,
                    "bm25_top7": [{"id": c, "relevant": c in g} for c in r]})
json.dump(samples, open(OUT_DIR / "p7_samples.json", "w"), indent=1, ensure_ascii=False)
print(json.dumps(samples[0], indent=1, ensure_ascii=False)[:2500])

{
 "query_id": "0000576325",
 "query": {
  "first": [
   "JUDGMENT <NAME> <NAME>, C.J. ",
   "1. The question referred to us for decision is-- ",
   "Is it open to an undischarged insolvent to maintain a suit regarding his after-acquired properties subject to the right of the Official Receiver to intervene in such proceedings? ",
   "2. In our view, upon the facts of this case this question does not arise and we are not disposed to enter into any further discussion of this question which is purely one of academic interest. ",
   "3. The facts of the case may be quite shortly stated. The insolvent was the plaintiff in <ORG>. He was also an undischarged insolvent. His case as set out in the plaint was that he got emeralds from one <NAME> for sale and that he gave them to the defendant in the suit for sale. The value of the eme…"
  ],
  "last": [
   "It is always to be remembered that, although goods in the hands of an agent may be easily distinguishable, they may yet, on the bankruptcy o

In [13]:
# Cell 13 — Save the statistics and print the headline numbers.
json.dump(stats, open(OUT_DIR / "p7_stats.json", "w"), indent=1, ensure_ascii=False)
print("Saved:", sorted(p.name for p in OUT_DIR.iterdir()))
print("queries:", stats["n_queries"], "| pool sizes:", stats["pool_size"])
for s in ("train", "dev"):
    print(f"{s}: relevant per query {stats['relevant'][s]['per_query']} | oracle F1@7 {stats['relevant'][s]['oracle_at_7']['f1']}")
print("query in own pool:", stats["query_in_own_pool"])
print("BM25 dev full text:", stats["bm25"]["dev_full_text"])

Saved: ['p7_samples.json', 'p7_stats.json']
queries: {'train': 827, 'dev': 118, 'test': 237} | pool sizes: {'train': 4320, 'dev': 1023, 'test': 1727}
train: relevant per query {'p0': 0.0, 'p10': 1.0, 'p25': 2.0, 'p50': 4.0, 'p75': 9.0, 'p90': 14.0, 'p95': 19.0, 'p99': 31.0, 'p100': 52.0, 'mean': 6.42} | oracle F1@7 0.6386
dev: relevant per query {'p0': 1.0, 'p10': 1.0, 'p25': 2.0, 'p50': 5.0, 'p75': 9.8, 'p90': 17.3, 'p95': 24.0, 'p99': 58.2, 'p100': 92.0, 'mean': 8.25} | oracle F1@7 0.6181
query in own pool: {'train': 827, 'dev': 118, 'test': 237}
BM25 dev full text: {'micro_p@7': 0.1392, 'micro_r@7': 0.1182, 'micro_f1@7': 0.1278, 'recall@7': 0.1182, 'recall@20': 0.1809, 'recall@50': 0.2569, 'recall@100': 0.3638, 'recall@200': 0.5447}
